In [ ]:
import geopandas as gpd
import matplotlib.pyplot as plt
from pathlib import Path
import folium
import pandas as pd

# Show the base graph network

In [ ]:
root_dir = Path("").parent
#file_path = Path("").parent / "base_graph_hazard_edges.gpkg"
path_csv = Path("").parent / "event_weights.csv"
#gdf =  gpd.read_file(file_path, driver = "GPKG")

file_path_result = Path("").parent / "result_gdf.json"
gdf = gpd.read_file(file_path_result, driver="JSON")
gdf


In [ ]:
# import the scenario names and their respective weights:
scenarios_weights_df = pd.read_csv(path_csv)
number_of_scenarios = len(scenarios_weights_df)


aggregate_wl = "max"  # this we should get from network_config or analysis_config

aggregate_wls = {
    "max": "ma",
    "mean": "me",
    "min": "mi"
}

aggregate_wl_column = aggregate_wls[aggregate_wl]

columns_to_count = []
columns_to_count_fraction = []
for name in scenarios_weights_df['name'].to_list():
    actual =  name + f"_{aggregate_wl_column}"
    actual_fraction = name + "_fr"
    if actual in gdf.columns:
        columns_to_count.append(actual)
        columns_to_count_fraction.append(actual_fraction)


weights_dict = {}
for _, row in scenarios_weights_df.iterrows():
    weights_dict[row['name'] + f"_{aggregate_wl_column}"] = row['weight']


## Modify the dataframe

In [ ]:
# Parametrization
threshold = 0.5  # would be nice to make this threshold into a slider.

fraction_threshold = 0.0

color_on_probability = False


In [ ]:
# Add a new column with the count of non-zero values across specified columns
gdf['count'] = 0
for i, row in gdf.iterrows():
    row_count = 0
    for hazard_depth, hazard_fraction in zip(columns_to_count, columns_to_count_fraction):
        if row[hazard_fraction] > fraction_threshold:
            if row[hazard_depth] > threshold:
                row_count += 1
    gdf.at[i, 'count'] = row_count


for index, row in gdf.iterrows():
    total_weight = 0
    for col in columns_to_count:
        if row[col] > threshold:
            # Find the corresponding weight and add to total
            weight = weights_dict[col]
            total_weight += weight
    gdf.at[index, 'cum_pf'] = total_weight

temp = gdf[gdf['count'] > 0]
temp




In [ ]:
import folium


def mute_basemap(map: folium.Map) -> folium.Map:
    map.get_root().header.add_child(
        folium.Element(
            "<style>.leaflet-tile-pane { filter: grayscale(1); opacity: 0.6; }</style>"
        ),
        name="muted_basemap",
    )
    return map


from branca.colormap import LinearColormap
 
# Create a Folium pop_diff_map centered around the mean of the geometry
center = [
    gdf['geometry'].centroid.y.mean(),
    gdf['geometry'].centroid.x.mean()
]
pop_diff_map = mute_basemap(folium.Map(location=center, zoom_start=5, control_scale=True, tiles="Esri.WorldStreetMap"))
 
 
if color_on_probability:
    var = 'cum_pf'
else:
    var = "count"

vmin=(gdf[var]).min()
vmax=(gdf[var]).max()
# Classify values into ranges
ranges = [vmin, vmin+((vmax-vmin)/4), vmin+2*((vmax-vmin)/4), vmin+3*((vmax-vmin)/4) , vmax]

ranges=sorted(ranges)
# Create a colormap from green to green to red using the overall min and max values
colormap = LinearColormap(colors=['lightgreen', 'orange', 'darkred'],
                          vmin=0,
                          vmax=vmax
                          )
 
colormap.caption = 'Affected people-Optimal routes length difference (people x km)'
 


   

 
    # Add target_optimal_routes_with_hazard_gdf with created Colormap to feature_group
for idx, row in gdf.iterrows():
    value = row['count']
    cum_pf = row['cum_pf']
    # color = 'blue' if (row['count'] > 0)  else 'lightgrey' 

    if row[var] == 0:
        color = 'lightgrey'
    else:
        color = colormap(row[var])

    # Extracting coordinates from MultiLineString
    coordinates = []
    if row['geometry'].geom_type == 'MultiLineString':
        for line in row['geometry']:
            coords = [(coord[1], coord[0]) for coord in line.coords]
            coordinates.extend(coords)
    else:
        coordinates = [(coord[1], coord[0]) for coord in row['geometry'].coords]
    # Create a popup with data
    popup_content = f"Nb scenario disrupting {value}/{len(columns_to_count)}, cumulative pf: {cum_pf}"
    popup = folium.Popup(popup_content, max_width=300)

    folium.PolyLine(
        locations=coordinates,
        color=color,
        weight=3,
        opacity=1,
        popup=popup
    ).add_to(pop_diff_map)


    #  from branca.colormap import LinearColormap


# colormap = branca.colormap.linear.YlOrRd_09.scale(0, 8500)
colormap = colormap.to_step(index=ranges)
colormap.caption = 'Number of scenarios for which link is disrupted'
colormap.add_to(pop_diff_map)

#pop_diff_map.save(root_dir / "map_figure.html")

pop_diff_map